make a new file that loads the h5 & csv with labels. drop rows in the csv that don't have a label or have DELETE as their label. drop rows from h5 that aren't in csv. output smaller h5 that's useable

In [1]:
import pandas as pd
import os
from pathlib import Path
import librosa
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import h5py
from tqdm import tqdm
import hashlib

In [2]:
# get file paths
FOLDER_PATH = "/Users/cheoso/Documents/samples/Packs/"
file_paths = []
total = 0

for root, dirs, files in os.walk(FOLDER_PATH):
    for file in files:
        path = Path(root) / file
        path_lower = str(path).lower()
        if path_lower.endswith(('.wav', '.aif', '.aiff')):
          file_paths.append(str(path))

          total += path.stat().st_size

print(f"Total of {total} bytes.")

Total of 84644864348 bytes.


In [16]:
failed_files = []
audio_hashes = set()
divided_by_zero = []

def process_audio(audio, sr):
    rs = librosa.resample(audio, orig_sr=sr, target_sr=44100)
    '''try:
        rs = rs / abs(rs.max())
    except ZeroDivisionError:
        print(f'division error. max value of {rs.max()}')
        divided_by_zero.append(sr)'''
    if rs.max() == 0:
        #print(f'rs.max() == 0')
        divided_by_zero.append(sr)
    else:
        return rs / abs(rs.max())
    return rs

def hash_waveform_with_id(row):
    waveform = row
    waveform_hash = hashlib.sha256(waveform.tobytes()).hexdigest()
    return waveform_hash

def extract_metadata(index, file):
    """Load an audio file and extract its waveform, sample rate, and label."""
    try:
        audio, sr = librosa.load(file, sr=None)  # Load without resampling
        audio = process_audio(audio, sr)
        audio = hash_waveform_with_id(audio)
        if audio in audio_hashes:
            return None
        else:
            audio_hashes.add(audio)
            file = str(file)
            file_name = file.split('/')[-1]
            return [index, file, file_name]
    except Exception as e:
        failed_files.append(file)
        return None

# Use parallel processing to load audio files
def get_audio_and_metadata(files):
    meta_results = []

    with ThreadPoolExecutor() as executor:
        results = executor.map(lambda p: extract_metadata(*p), enumerate(files))
        
        for res in results:
            if res is not None:
                meta_data = res
                meta_results.append(meta_data)
    
    return meta_results


metadataResults = get_audio_and_metadata(file_paths)

/var/folders/h0/pzm07l6x5n7181rc09xppl940000gn/T/ipykernel_38938/2493045814.py:27: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file, sr=None)  # Load without resampling
/Users/cheoso/ai_projects/load_audio/venv/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


rs.max() == 0
rs.max() == 0
rs.max() == 0
rs.max() == 0
rs.max() == 0
rs.max() == 0
rs.max() == 0
rs.max() == 0
rs.max() == 0
rs.max() == 0
rs.max() == 0
rs.max() == 0


In [17]:
failed_len = len(failed_files)
print(failed_len)
print(len(divided_by_zero))

639
12


In [18]:
print(len(audio_hashes))

51159


In [19]:
print(len(file_paths))

52910


In [20]:
#print(len(audioResults))
#old 52271
print(len(metadataResults))

51159


In [24]:
#metadataResults[:2]

In [11]:
# Convert results to DataFrame
#df_ar = pd.DataFrame(audioResults, columns=["id", "sample_rate", "waveform"])
df_md = pd.DataFrame(metadataResults, columns=["id", "file_path", "file_name"])

In [ ]:
#df_md.filter()

In [8]:
'''dt = h5py.string_dtype(encoding='utf-8')


with h5py.File("audio_data.h5", "w") as f:
    meta_data = f.create_group("meta_data")
    meta_data.create_dataset('index', data=np.array(df_md['id'], dtype='int'))
    meta_data.create_dataset('path', data=np.array(df_md['file_path'], dtype=dt))
    meta_data.create_dataset('name', data=np.array(df_md['file_name'], dtype=dt))

    audio_data = f.create_group('audio_data')

    def save_sample(row):
        sample_group = audio_data.create_group(str(row['id']))
        sample_group.create_dataset('waveform', data=row['waveform'], compression='gzip')
        sample_group.create_dataset('sample_rate', data=row['sample_rate'])

    # Use ThreadPoolExecutor for parallelism
    with ThreadPoolExecutor(max_workers=8) as executor:
        list(tqdm(executor.map(save_sample, [row for _, row in df_ar.iterrows()]), total=len(df_ar)))'''
    

'dt = h5py.string_dtype(encoding=\'utf-8\')\n\n\nwith h5py.File("audio_data.h5", "w") as f:\n    meta_data = f.create_group("meta_data")\n    meta_data.create_dataset(\'index\', data=np.array(df_md[\'id\'], dtype=\'int\'))\n    meta_data.create_dataset(\'path\', data=np.array(df_md[\'file_path\'], dtype=dt))\n    meta_data.create_dataset(\'name\', data=np.array(df_md[\'file_name\'], dtype=dt))\n\n    audio_data = f.create_group(\'audio_data\')\n\n    def save_sample(row):\n        sample_group = audio_data.create_group(str(row[\'id\']))\n        sample_group.create_dataset(\'waveform\', data=row[\'waveform\'], compression=\'gzip\')\n        sample_group.create_dataset(\'sample_rate\', data=row[\'sample_rate\'])\n\n    # Use ThreadPoolExecutor for parallelism\n    with ThreadPoolExecutor(max_workers=8) as executor:\n        list(tqdm(executor.map(save_sample, [row for _, row in df_ar.iterrows()]), total=len(df_ar)))'

In [ ]:
df_md.to_csv("audio_metadata.csv", header=True, index=False, encoding='utf-8-sig')

In [39]:
df_md.head(5)

,id,file_path,file_name
0,0,/Users/cheoso/Documents/samples/Packs/Heavywei...,EO_HVYWT_JNGL_D_808_Dist_Synth_Bass.wav
1,1,/Users/cheoso/Documents/samples/Packs/Heavywei...,EO_HVYWT_JNGL_D_808_Deep_Synth_Bass.wav
2,2,/Users/cheoso/Documents/samples/Packs/Heavywei...,EO_HVYWT_JNGL_D_Dist_Wub_Synth_Bass_02.wav
3,3,/Users/cheoso/Documents/samples/Packs/Heavywei...,EO_HVYWT_JNGL_D_Hard_808_Synth_Bass_02.wav
4,4,/Users/cheoso/Documents/samples/Packs/Heavywei...,EO_HVYWT_JNGL_D_Donk_Pluck_Synth_Bass_02.wav
